In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
import scipy.sparse.linalg as spla
from functools import reduce

# Notes

In [ ]:
np.arange(16).reshape((4, 4))

In [ ]:
def fdm_neg_lapl_1d_mat(interior_size, is_low_bd_nd, is_high_bd_nd):
    return sparse.diags_array(
        [2.0, -1, -1], offsets=[0, 1, -1], shape=(interior_size, interior_size)
    )


def fdm_neg_lapl_unif_disc_mat(interior_shape, non_dirichlet_bd=None):
    num_dims = len(interior_shape)
    if non_dirichlet_bd is None:
        non_dirichlet_bd = num_dims * ((False, False),)

    assert len(non_dirichlet_bd) == num_dims, f"mismatch in dimensions"

    return reduce(
        sparse.kronsum,
        (
            fdm_neg_lapl_1d_mat(dim_size, is_low_bd_nd, is_high_bd_nd)
            for dim_size, (is_low_bd_nd, is_high_bd_nd) in zip(
                # KLUDGE: scipy kronsum swapped the order that matrices get kron'd
                # so we're reversing the stuff here to correct for it
                reversed(interior_shape),
                reversed(non_dirichlet_bd),
            )
        ),
    )


fig, ax = plt.subplots()
ax: plt.Axes = ax

ax_im = ax.matshow(fdm_neg_lapl_unif_disc_mat((5, 4, 3)).toarray())
fig.colorbar(ax_im)